# Programación orientada a objetos con Python
## Video 3. Métodos de instancia, de clase (`@classmethod`) y estáticos (`@staticmethod`)

**Hilo conductor de la serie:** construiremos objetos matemáticos (fracciones, vectores y polinomios) y los iremos ampliando de video en video.

**Al terminar este video podrás:**
1. Explicar por qué un único inicializador `__init__` no basta cuando un objeto matemático admite varias representaciones de entrada.
2. Construir métodos de clase con `@classmethod` y el parámetro `cls` para definir constructores alternativos y constantes notables.
3. Definir métodos estáticos con `@staticmethod` para agrupar funciones matemáticas auxiliares dentro del espacio de nombres de la clase.
4. Elegir con criterio preciso cuándo usar un método de instancia, un método de clase o un método estático.

## 1. Planteamiento del problema

**Definición.** Un vector del plano euclidiano $\mathbb{R}^2$ queda determinado por sus *coordenadas cartesianas* $(x, y) \in \mathbb{R}^2$, o bien por sus *coordenadas polares* $(r, \theta)$ con radio $r \ge 0$ y ángulo $\theta \in (-\pi, \pi]$, ligadas mediante las relaciones trigonométricas

$$x = r \cos\theta, \qquad y = r \sin\theta.$$

De manera análoga, un número racional en $\mathbb{Q}$ puede describirse como un par ordenado de enteros $(p, q)$ con $q \neq 0$, como un único número entero $n = \frac{n}{1}$ o como una cadena de texto de la forma `"p/q"`.

**Problema: múltiples formas de construir un mismo objeto matemático y organización de funciones auxiliares.**

- **Entrada:** datos provenientes de representaciones distintas de un mismo objeto (coordenadas cartesianas $(x, y)$ o polares $(r, \theta)$ para `VectorPlano`; un par $(p, q)$, un entero $n$ o una cadena `"p/q"` para `Fraccion`).
- **Salida:** instancias válidas creadas mediante constructores con nombre explícito sin sobrecargar ni volver ambiguo a `__init__`, junto con funciones auxiliares puras (como el algoritmo de Euclides o la simplificación de ceros en listas de coeficientes) alojadas dentro de la clase.

## 2. El límite de un único `__init__` y los métodos de instancia

En Python cada clase tiene un único método `__init__`. Si `VectorPlano.__init__` recibe dos números reales $(a, b)$, es imposible distinguir por sus tipos si quien llama al constructor quiso pasar coordenadas cartesianas $(x, y)$ o coordenadas polares $(r, \theta)$.

Recordemos qué es un **método de instancia**: una función definida dentro de la clase cuyo primer parámetro es `self` (la instancia ya existente). A los métodos de instancia les corresponden todas las operaciones que dependen del estado particular de un objeto ya creado, como `norma`, `angulo`, `escalar` y `sumar`.

In [ ]:
from math import atan2, cos, pi, sin, sqrt


class VectorPlano:
    dimension = 2

    def __init__(self, x, y):
        self.x = float(x)
        self.y = float(y)

    def norma(self):
        return sqrt(self.x ** 2 + self.y ** 2)

    def angulo(self):
        return atan2(self.y, self.x)

    def escalar(self, factor):
        return VectorPlano(factor * self.x, factor * self.y)

    def sumar(self, otro):
        return VectorPlano(self.x + otro.x, self.y + otro.y)

    def mostrar(self):
        return f"({self.x}, {self.y})"


u = VectorPlano(1.0, sqrt(3.0))
print("u =", u.mostrar())
print("Norma de u:", u.norma())
print("Ángulo de u en radianes:", round(u.angulo(), 6), "| pi/3 =", round(pi / 3, 6))
print("Vector 2*u:", u.escalar(2).mostrar())

## 3. Métodos de clase (`@classmethod`): constructores alternativos

**Definición.** Un *método de clase* es un método precedido por el decorador `@classmethod`. Su primer parámetro, llamado por convención `cls`, recibe automáticamente la clase misma (en lugar de una instancia) tanto si se invoca como `Clase.metodo(...)` como si se invoca desde una instancia.

Por esa razón, `@classmethod` es la herramienta idónea para definir **constructores alternativos** y **objetos notables**:

- `VectorPlano.desde_polares(radio, angulo)` valida $r \ge 0$, calcula $(r\cos\theta, r\sin\theta)$ y devuelve `cls(x, y)`.
- `VectorPlano.origen()` devuelve `cls(0.0, 0.0)` y `VectorPlano.base_canonica()` devuelve la pareja `(cls(1.0, 0.0), cls(0.0, 1.0))`.
- `Fraccion.desde_entero(entero)` devuelve `cls(entero, 1)` y `Fraccion.desde_cadena(texto)` analiza cadenas como `"-15/20"` o `"7"` y devuelve la instancia normalizada de `cls`.

Llamar a `cls(...)` en lugar de escribir el nombre fijo de la clase garantiza que, cuando estudiemos herencia en el video 6, las subclases construyan instancias de su propio tipo.

In [ ]:
class VectorPlano:
    dimension = 2

    def __init__(self, x, y):
        self.x = float(x)
        self.y = float(y)

    @classmethod
    def origen(cls):
        return cls(0.0, 0.0)

    @classmethod
    def base_canonica(cls):
        return cls(1.0, 0.0), cls(0.0, 1.0)

    @classmethod
    def desde_polares(cls, radio, angulo):
        if radio < 0:
            raise ValueError("El radio en coordenadas polares no puede ser negativo.")
        return cls(radio * cos(angulo), radio * sin(angulo))

    def norma(self):
        return sqrt(self.x ** 2 + self.y ** 2)

    def angulo(self):
        return atan2(self.y, self.x)

    def escalar(self, factor):
        return VectorPlano(factor * self.x, factor * self.y)

    def sumar(self, otro):
        return VectorPlano(self.x + otro.x, self.y + otro.y)

    def mostrar(self):
        return f"({self.x}, {self.y})"


v = VectorPlano.desde_polares(2.0, pi / 3)
e1, e2 = VectorPlano.base_canonica()
cero = VectorPlano.origen()

print("Desde polares (2, pi/3):", v.mostrar())
print("Norma de v:", round(v.norma(), 6), "| Ángulo:", round(v.angulo(), 6))
print("Base canónica:", e1.mostrar(), "y", e2.mostrar())
print("Origen:", cero.mostrar())

## 4. Métodos estáticos (`@staticmethod`): funciones auxiliares dentro de la clase

**Definición.** Un *método estático* es una función definida dentro de una clase precedida por el decorador `@staticmethod`. No recibe `self` ni `cls`; su firma y su comportamiento son los de una función ordinaria, pero se aloja dentro de la clase porque su propósito está estrechamente ligado a ella.

Un ejemplo clásico en nuestra clase `Fraccion` es el **Algoritmo de Euclides** para calcular el máximo común divisor sin depender de funciones externas:

- **Definición.** Dados dos números enteros $a, b \in \mathbb{Z}$ no ambos nulos, su *máximo común divisor*, denotado $\gcd(a, b)$, es el mayor entero positivo que divide simultáneamente a $a$ y a $b$.
- **Entrada:** dos números enteros $a, b \in \mathbb{Z}$ con $(a, b) \neq (0, 0)$.
- **Salida:** el entero positivo $d = \gcd(a, b)$ obtenido aplicando reiteradamente la identidad $\gcd(a, b) = \gcd(b, a \bmod b)$ hasta que el segundo término sea cero.

In [ ]:
class Fraccion:
    cuerpo = "Q"

    def __init__(self, numerador, denominador):
        if denominador == 0:
            raise ValueError("El denominador no puede ser cero.")
        if denominador < 0:
            numerador, denominador = -numerador, -denominador
        divisor = self.maximo_comun_divisor(numerador, denominador)
        self.numerador = numerador // divisor
        self.denominador = denominador // divisor

    @staticmethod
    def maximo_comun_divisor(a, b):
        if a == 0 and b == 0:
            raise ValueError("Al menos uno de los enteros debe ser distinto de cero.")
        a, b = abs(a), abs(b)
        while b != 0:
            a, b = b, a % b
        return a

    @staticmethod
    def minimo_comun_multiplo(a, b):
        if a == 0 or b == 0:
            return 0
        return abs(a) // Fraccion.maximo_comun_divisor(a, b) * abs(b)

    @classmethod
    def cero(cls):
        return cls(0, 1)

    @classmethod
    def uno(cls):
        return cls(1, 1)

    @classmethod
    def desde_entero(cls, entero):
        return cls(entero, 1)

    @classmethod
    def desde_cadena(cls, texto):
        texto_limpio = texto.strip()
        if "/" in texto_limpio:
            partes = texto_limpio.split("/")
            if len(partes) != 2:
                raise ValueError(f"Formato de fracción inválido: {texto!r}")
            numerador = int(partes[0].strip())
            denominador = int(partes[1].strip())
            return cls(numerador, denominador)
        return cls(int(texto_limpio), 1)

    def sumar(self, otra):
        return Fraccion(
            self.numerador * otra.denominador + self.denominador * otra.numerador,
            self.denominador * otra.denominador,
        )

    def multiplicar(self, otra):
        return Fraccion(
            self.numerador * otra.numerador,
            self.denominador * otra.denominador,
        )

    def mostrar(self):
        return f"{self.numerador}/{self.denominador}"


print("Máximo común divisor de 84 y 30:", Fraccion.maximo_comun_divisor(84, 30))
print("Mínimo común múltiplo de 12 y 18:", Fraccion.minimo_comun_multiplo(12, 18))

f1 = Fraccion.desde_cadena(" -15 / 20 ")
f2 = Fraccion.desde_entero(2)
print("Desde cadena '-15 / 20':", f1.mostrar())
print("Desde entero 2:", f2.mostrar())
print("Suma:", f1.sumar(f2).mostrar())
print("Cero y uno:", Fraccion.cero().mostrar(), "y", Fraccion.uno().mostrar())

### Tabla comparativa y regla práctica de elección

| Criterio | Método de instancia | Método de clase (`@classmethod`) | Método estático (`@staticmethod`) |
|---|---|---|---|
| Decorador | Ninguno | `@classmethod` | `@staticmethod` |
| Primer parámetro | `self` (la instancia) | `cls` (la clase) | Ninguno implícito |
| Estado al que accede | Atributos propios de la instancia y atributos de la clase | Atributos y constructores de la clase | Solo sus parámetros explícitos |
| Forma habitual de llamada | `objeto.metodo(...)` | `Clase.metodo(...)` | `Clase.metodo(...)` |
| Uso principal en objetos matemáticos | Operaciones sobre un objeto ya construido (`sumar`, `norma`, `evaluar`) | Constructores alternativos y constantes notables (`desde_polares`, `desde_cadena`, `cero`) | Funciones auxiliares puras relacionadas con el dominio (`maximo_comun_divisor`, `quitar_ceros_finales`) |

## 5. Ejercicios

**Ejercicio 1 (coprimalidad y media aritmética en `Fraccion`).** Añade a la clase `Fraccion`:
1. Un método estático `son_coprimos(a, b)`:
   - **Entrada:** dos números enteros $a, b \in \mathbb{Z}$ no ambos nulos.
   - **Salida:** `True` si $\gcd(a, b) = 1$, y `False` en caso contrario.
2. Un método de clase `punto_medio(cls, primera, segunda)`:
   - **Entrada:** dos instancias `primera` y `segunda` de `Fraccion`.
   - **Salida:** una nueva instancia de `Fraccion` igual a la media aritmética $\frac{\text{primera} + \text{segunda}}{2}$.

**Ejercicio 2 (vector unitario direccional y determinante en `VectorPlano`).** Añade a la clase `VectorPlano`:
1. Un método de clase `unitario_en_direccion(cls, angulo)`:
   - **Entrada:** un ángulo $\theta \in \mathbb{R}$ en radianes.
   - **Salida:** la instancia de `VectorPlano` correspondiente al vector unitario $(\cos\theta, \sin\theta)$.
2. Un método estático `determinante_2x2(u, v)`:
   - **Entrada:** dos instancias $u = (x_1, y_1)$ y $v = (x_2, y_2)$ de `VectorPlano`.
   - **Salida:** el número real $x_1 y_2 - y_1 x_2$, que representa el área orientada del paralelogramo generado por $u$ y $v$.

**Ejercicio 3 (constructores, producto y derivada formal en `Polinomio`).** Amplía la clase `Polinomio` integrando los tres tipos de métodos:
1. Un método estático `quitar_ceros_finales(coeficientes)` que reciba una secuencia no vacía de números y devuelva una lista nueva sin los ceros finales redundantes (conservando `[0]` para el polinomio nulo). Úsalo dentro de `__init__`.
2. Un método de clase `monomio(cls, coeficiente, grado)` que construya el monomio $c\,x^k$ para $c \in \mathbb{R}$ y un entero $k \ge 0$.
3. Un método de instancia `multiplicar(self, otro)` y un método de instancia `derivar(self)`:
   - **Entrada de `derivar`:** el polinomio $P(x) = \sum_{k=0}^{n} a_k x^k$.
   - **Salida de `derivar`:** el polinomio derivado $P'(x) = \sum_{k=1}^{n} k\,a_k x^{k-1}$, o el polinomio nulo si $n = 0$.
4. Un método de clase `desde_raices(cls, raices)`:
   - **Entrada:** una secuencia de números reales $[r_1, r_2, \dots, r_m]$.
   - **Salida:** el polinomio mónico $\prod_{k=1}^{m} (x - r_k)$ obtenido multiplicando sucesivamente los factores lineales $(-r_k) + 1\cdot x$.

## 6. Soluciones

In [ ]:
# Ejercicio 1
class Fraccion:
    cuerpo = "Q"

    def __init__(self, numerador, denominador):
        if denominador == 0:
            raise ValueError("El denominador no puede ser cero.")
        if denominador < 0:
            numerador, denominador = -numerador, -denominador
        divisor = self.maximo_comun_divisor(numerador, denominador)
        self.numerador = numerador // divisor
        self.denominador = denominador // divisor

    @staticmethod
    def maximo_comun_divisor(a, b):
        if a == 0 and b == 0:
            raise ValueError("Al menos uno de los enteros debe ser distinto de cero.")
        a, b = abs(a), abs(b)
        while b != 0:
            a, b = b, a % b
        return a

    @staticmethod
    def son_coprimos(a, b):
        return Fraccion.maximo_comun_divisor(a, b) == 1

    @classmethod
    def punto_medio(cls, primera, segunda):
        suma = primera.sumar(segunda)
        return cls(suma.numerador, suma.denominador * 2)

    def sumar(self, otra):
        return Fraccion(
            self.numerador * otra.denominador + self.denominador * otra.numerador,
            self.denominador * otra.denominador,
        )

    def mostrar(self):
        return f"{self.numerador}/{self.denominador}"


print("¿14 y 15 son coprimos?", Fraccion.son_coprimos(14, 15))
print("¿14 y 21 son coprimos?", Fraccion.son_coprimos(14, 21))
media = Fraccion.punto_medio(Fraccion(1, 3), Fraccion(1, 2))
print("Punto medio entre 1/3 y 1/2:", media.mostrar())

In [ ]:
# Ejercicio 2
class VectorPlano:
    dimension = 2

    def __init__(self, x, y):
        self.x = float(x)
        self.y = float(y)

    @classmethod
    def desde_polares(cls, radio, angulo):
        if radio < 0:
            raise ValueError("El radio en coordenadas polares no puede ser negativo.")
        return cls(radio * cos(angulo), radio * sin(angulo))

    @classmethod
    def unitario_en_direccion(cls, angulo):
        return cls.desde_polares(1.0, angulo)

    @staticmethod
    def determinante_2x2(u, v):
        return u.x * v.y - u.y * v.x

    def norma(self):
        return sqrt(self.x ** 2 + self.y ** 2)

    def mostrar(self):
        return f"({self.x}, {self.y})"


unitario = VectorPlano.unitario_en_direccion(pi / 4)
print("Vector unitario en pi/4:", (round(unitario.x, 6), round(unitario.y, 6)))
print("Norma del vector unitario:", round(unitario.norma(), 6))
print("Determinante de (2, 0) y (0, 3):", VectorPlano.determinante_2x2(VectorPlano(2, 0), VectorPlano(0, 3)))

In [ ]:
# Ejercicio 3
class Polinomio:
    variable = "x"

    def __init__(self, coeficientes):
        self.coeficientes = self.quitar_ceros_finales(coeficientes)

    @staticmethod
    def quitar_ceros_finales(coeficientes):
        lista = list(coeficientes)
        if not lista:
            raise ValueError("La secuencia de coeficientes no puede estar vacía.")
        while len(lista) > 1 and lista[-1] == 0:
            lista.pop()
        return lista

    @classmethod
    def monomio(cls, coeficiente, grado):
        if grado < 0:
            raise ValueError("El grado del monomio no puede ser negativo.")
        return cls([0] * grado + [coeficiente])

    @classmethod
    def desde_raices(cls, raices):
        resultado = cls([1])
        for raiz in raices:
            resultado = resultado.multiplicar(cls([-raiz, 1]))
        return resultado

    def grado(self):
        return len(self.coeficientes) - 1

    def evaluar(self, valor):
        resultado = 0
        for coeficiente in reversed(self.coeficientes):
            resultado = resultado * valor + coeficiente
        return resultado

    def multiplicar(self, otro):
        producto = [0] * (self.grado() + otro.grado() + 1)
        for i, a in enumerate(self.coeficientes):
            for j, b in enumerate(otro.coeficientes):
                producto[i + j] += a * b
        return type(self)(producto)

    def derivar(self):
        if self.grado() == 0:
            return type(self)([0])
        derivados = [
            exponente * coeficiente
            for exponente, coeficiente in enumerate(self.coeficientes)
            if exponente > 0
        ]
        return type(self)(derivados)

    def mostrar(self):
        return " + ".join(
            f"{coeficiente}*{self.variable}^{exponente}"
            for exponente, coeficiente in enumerate(self.coeficientes)
        )


monomio = Polinomio.monomio(4, 3)
p = Polinomio.desde_raices([1, 2, 3])
derivada = p.derivar()

print("Monomio 4x^3:", monomio.coeficientes)
print("Polinomio con raíces 1, 2, 3:", p.mostrar())
print("Evaluación en 1, 2 y 3:", p.evaluar(1), p.evaluar(2), p.evaluar(3))
print("Derivada P'(x):", derivada.mostrar())

## 7. Resumen

- Los **métodos de instancia** reciben `self` y operan sobre el estado de un objeto ya creado.
- Los **métodos de clase** (`@classmethod`) reciben `cls` y permiten construir instancias desde representaciones alternativas (`desde_polares`, `desde_cadena`, `desde_raices`) o devolver constantes notables sin acoplar la lógica al nombre fijo de la clase.
- Los **métodos estáticos** (`@staticmethod`) no reciben `self` ni `cls`; organizan funciones auxiliares puras (como `maximo_comun_divisor` o `quitar_ceros_finales`) dentro de la clase a la que pertenecen conceptualmente.
- En el siguiente video protegeremos las reglas internas de nuestros objetos mediante encapsulamiento: convención de privacidad, `@property` e invariantes.